In [31]:
import json
import time
import sqlite3
import requests
import pandas as pd

session = requests.Session()

def get_json(url, params=None, tries=5):
    """Get JSON, waiting and retrying if the API pushes back."""
    for attempt in range(tries):
        try:
            r = session.get(url, params=params, timeout=20)
            if r.status_code == 200:
                return r.json()
            print(f"Status {r.status_code} - ", end="")
        except (requests.RequestException, ValueError) as e:
            print(f"Error ({type(e).__name__}) - ", end="")
        wait = 5 * (attempt + 1)
        print(f"waiting {wait}s and retrying")
        time.sleep(wait)
    raise RuntimeError(f"Gave up on {url}")

# --- Step 1: every NHL skater ID since 2011-12 ---
STATS_URL = "https://api.nhle.com/stats/rest/en/skater/summary"
player_ids = set()

for year in range(2000, 2026):
    season = int(f"{year}{year + 1}")          # e.g. 20112012
    start = 0
    while True:
        params = {
            "start": start, "limit": 100,
            "sort": json.dumps([{"property": "playerId", "direction": "ASC"}]),
            "cayenneExp": f"seasonId={season} and gameTypeId=2",
        }
        data = get_json(STATS_URL, params)
        player_ids.update(p["playerId"] for p in data["data"])
        start += 100
        time.sleep(0.5)
        if start >= data["total"]:
            break
    print(season, "done, unique players so far:", len(player_ids))

# --- Step 2: pull each player's career, keep OHL + NHL regular seasons ---
rows = []
for i, pid in enumerate(sorted(player_ids)):
    try:
        p = get_json(f"https://api-web.nhle.com/v1/player/{pid}/landing")
    except RuntimeError as e:
        print("Skipping", pid, "-", e)
        continue

    name = f'{p["firstName"]["default"]} {p["lastName"]["default"]}'
    for s in p.get("seasonTotals", []):
        if s["gameTypeId"] == 2:
            rows.append({
                "player_id": pid,
                "name": name,
                "position": p.get("position"),
                "birth_date": p.get("birthDate"),
                "season": s["season"],
                "league": s["leagueAbbrev"],
                "team": s["teamName"]["default"],
                "gp": s.get("gamesPlayed", 0),
                "goals": s.get("goals", 0),
                "assists": s.get("assists", 0),
                "points": s.get("points", 0),
            })

    if i % 200 == 0:
        print(f"{i}/{len(player_ids)} players pulled")
    time.sleep(0.25)

df = pd.DataFrame(rows)
conn = sqlite3.connect(r"C:\Users\flsto\OneDrive\Desktop\Python Projects\NHLe-model\hockey.db")
df.to_sql("career_seasons", conn, if_exists="replace", index=False)

query = """
WITH ohl AS (
    SELECT player_id, name, position, season, SUM(gp) AS gp, SUM(points) AS pts
    FROM career_seasons WHERE league = 'OHL'
    GROUP BY player_id, season
),
nhl AS (
    SELECT player_id, season, SUM(gp) AS gp, SUM(points) AS pts
    FROM career_seasons WHERE league = 'NHL'
    GROUP BY player_id, season
)
SELECT o.player_id, o.name, o.position, o.season AS ohl_season,
       o.gp AS ohl_gp, o.pts AS ohl_pts,
       n.gp AS nhl_gp, n.pts AS nhl_pts
FROM ohl o
JOIN nhl n
  ON o.player_id = n.player_id
 AND n.season = o.season + 10001
WHERE o.gp >= 20 AND n.gp >= 20
"""
pairs = pd.read_sql(query, conn)
pairs["ohl_ppg"] = pairs["ohl_pts"] / pairs["ohl_gp"]
pairs["nhl_ppg"] = pairs["nhl_pts"] / pairs["nhl_gp"]

def nhle(df):
    """Pooled factor: total NHL PPG divided by total OHL PPG."""
    return (df["nhl_pts"].sum() / df["nhl_gp"].sum()) / (df["ohl_pts"].sum() / df["ohl_gp"].sum())

print("Players in sample:", len(pairs))
print("OHL NHLe:", round(nhle(pairs), 3))

# Forwards vs defense
pairs["pos_group"] = pairs["position"].map(lambda p: "D" if p == "D" else "F")
print(pairs.groupby("pos_group").apply(nhle).round(3))

# Stability by season
print(pairs.groupby("ohl_season").apply(nhle).round(3))

20002001 done, unique players so far: 884
20012002 done, unique players so far: 1042
20022003 done, unique players so far: 1173
20032004 done, unique players so far: 1325
20042005 done, unique players so far: 1325
20052006 done, unique players so far: 1516
20062007 done, unique players so far: 1640
20072008 done, unique players so far: 1770
20082009 done, unique players so far: 1925
20092010 done, unique players so far: 2049
20102011 done, unique players so far: 2181
20112012 done, unique players so far: 2304
20122013 done, unique players so far: 2413
20132014 done, unique players so far: 2551
20142015 done, unique players so far: 2679
20152016 done, unique players so far: 2829
20162017 done, unique players so far: 2977
20172018 done, unique players so far: 3101
20182019 done, unique players so far: 3233
20192020 done, unique players so far: 3348
20202021 done, unique players so far: 3463
20212022 done, unique players so far: 3617
20222023 done, unique players so far: 3721
20232024 don

In [32]:
#Forward vs. Defense NHLe gap: 95% CI via bootstrapping

import numpy as np

def bootstrap_nhle(df, n_boot=1000, seed=42):
    """Resample players with replacement and recompute NHLe each time."""
    rng = np.random.default_rng(seed)
    results = []
    for _ in range(n_boot):
        sample = df.sample(n=len(df), replace=True, random_state=rng)
        results.append(nhle(sample))
    return np.array(results)

forwards = pairs[pairs["pos_group"] == "F"]
defense = pairs[pairs["pos_group"] == "D"]

for label, group in [("All", pairs), ("Forwards", forwards), ("Defense", defense)]:
    boots = bootstrap_nhle(group)
    low, high = np.percentile(boots, [2.5, 97.5])
    print(f"{label:9} n={len(group):4}  NHLe={nhle(group):.3f}  95% CI: [{low:.3f}, {high:.3f}]")

# Is the D - F gap real? Bootstrap the difference directly
diff = bootstrap_nhle(defense) - bootstrap_nhle(forwards)
low, high = np.percentile(diff, [2.5, 97.5])
print(f"\nD minus F: {nhle(defense) - nhle(forwards):.3f}  95% CI: [{low:.3f}, {high:.3f}]")

All       n= 215  NHLe=0.277  95% CI: [0.259, 0.297]
Forwards  n= 161  NHLe=0.272  95% CI: [0.252, 0.293]
Defense   n=  54  NHLe=0.300  95% CI: [0.259, 0.344]

D minus F: 0.028  95% CI: [-0.019, 0.074]


In [33]:
def build_pairs(league, min_gp=20, start_season=20002001):
    query = f"""
    WITH src AS (
        SELECT player_id, name, position, season, SUM(gp) AS gp, SUM(points) AS pts
        FROM career_seasons WHERE league = '{league}'
        GROUP BY player_id, season
    ),
    nhl AS (
        SELECT player_id, season, SUM(gp) AS gp, SUM(points) AS pts
        FROM career_seasons WHERE league = 'NHL'
        GROUP BY player_id, season
    )
    SELECT s.player_id, s.name, s.position, s.season AS src_season,
           s.gp AS ohl_gp, s.pts AS ohl_pts, n.gp AS nhl_gp, n.pts AS nhl_pts
    FROM src s
    JOIN nhl n ON s.player_id = n.player_id AND n.season = s.season + 10001
    WHERE s.gp >= {min_gp} AND n.gp >= {min_gp} AND s.season >= {start_season}
    """
    return pd.read_sql(query, conn)

# OHL pairs, so the age and team-adjustment cells still work
pairs = build_pairs("OHL").rename(columns={"src_season": "ohl_season"})
pairs["ohl_ppg"] = pairs["ohl_pts"] / pairs["ohl_gp"]
pairs["nhl_ppg"] = pairs["nhl_pts"] / pairs["nhl_gp"]
pairs["pos_group"] = pairs["position"].map(lambda p: "D" if p == "D" else "F")

# NHLe for each CHL league
for league in ["OHL", "WHL", "QMJHL"]:
    p = build_pairs(league)
    low, high = np.percentile(bootstrap_nhle(p), [2.5, 97.5])
    print(f"{league:6} n={len(p):4}  NHLe={nhle(p):.3f}  95% CI: [{low:.3f}, {high:.3f}]")

OHL    n= 137  NHLe=0.290  95% CI: [0.271, 0.311]
WHL    n= 115  NHLe=0.296  95% CI: [0.266, 0.325]
QMJHL  n=  50  NHLe=0.289  95% CI: [0.255, 0.325]


In [36]:
#age buckets comparison

births = pd.read_sql("SELECT DISTINCT player_id, birth_date FROM career_seasons", conn)

def age_breakdown(league, bins=(17, 18, 19), top_label=None):
    p = build_pairs(league).merge(births, on="player_id", how="left")
    start_year = p["src_season"] // 10000
    cutoff = pd.to_datetime(start_year.astype(str) + "-09-15")
    p["age"] = ((cutoff - pd.to_datetime(p["birth_date"])).dt.days / 365.25).astype(int)

    top = max(bins) + 1
    p["age_group"] = p["age"].clip(lower=min(bins), upper=top).astype(str)
    p.loc[p["age"] <= min(bins), "age_group"] = f"<={min(bins)}"
    p.loc[p["age"] >= top, "age_group"] = f"{top}+"

    print(f"--- {league} ---")
    for age, group in p.groupby("age_group"):
        if len(group) < 10:
            print(f"  Age {age:5}  n={len(group):4}  too few")
            continue
        low, high = np.percentile(bootstrap_nhle(group), [2.5, 97.5])
        print(f"  Age {age:5}  n={len(group):4}  NHLe={nhle(group):.3f}  95% CI: [{low:.3f}, {high:.3f}]")

# Junior leagues: young bins
for league in ["OHL", "WHL", "QMJHL"]:
    age_breakdown(league, bins=(17, 18, 19))

# Pro/college leagues: older bins
for league in ["AHL", "NCAA", "SHL", "Liiga"]:   # use the labels from your check
    age_breakdown(league, bins=(17, 18, 19, 20, 21, 22, 23))

--- OHL ---
  Age 18     n=  48  NHLe=0.267  95% CI: [0.231, 0.301]
  Age 19     n=  54  NHLe=0.260  95% CI: [0.232, 0.289]
  Age 20+    n=   4  too few
  Age <=17   n=  31  NHLe=0.361  95% CI: [0.326, 0.396]
--- WHL ---
  Age 18     n=  40  NHLe=0.295  95% CI: [0.243, 0.347]
  Age 19     n=  60  NHLe=0.290  95% CI: [0.249, 0.334]
  Age 20+    n=   3  too few
  Age <=17   n=  12  NHLe=0.315  95% CI: [0.254, 0.376]
--- QMJHL ---
  Age 18     n=  17  NHLe=0.275  95% CI: [0.237, 0.323]
  Age 19     n=  24  NHLe=0.267  95% CI: [0.224, 0.315]
  Age <=17   n=   9  too few
--- AHL ---
  Age 18     n=  16  NHLe=0.554  95% CI: [0.468, 0.675]
  Age 19     n= 111  NHLe=0.645  95% CI: [0.586, 0.708]
  Age 20     n= 391  NHLe=0.574  95% CI: [0.547, 0.605]
  Age 21     n= 484  NHLe=0.544  95% CI: [0.517, 0.571]
  Age 22     n= 468  NHLe=0.475  95% CI: [0.451, 0.499]
  Age 23     n= 297  NHLe=0.450  95% CI: [0.421, 0.483]
  Age 24+    n= 772  NHLe=0.396  95% CI: [0.380, 0.413]
--- NCAA ---
  Age 18  

In [23]:
#Checking for season stability, especially because it's been interesting lately
for season, group in pairs.groupby("ohl_season"):
    if len(group) < 5:
        print(f"{season}  n={len(group):3}  too few")
        continue
    low, high = np.percentile(bootstrap_nhle(group, n_boot=500), [2.5, 97.5])
    print(f"{season}  n={len(group):3}  NHLe={nhle(group):.3f}  95% CI: [{low:.3f}, {high:.3f}]")

19811982  n=  3  too few
19821983  n=  5  NHLe=0.404  95% CI: [0.269, 0.612]
19831984  n=  4  too few
19841985  n=  3  too few
19851986  n=  3  too few
19861987  n=  3  too few
19871988  n=  2  too few
19881989  n=  2  too few
19891990  n=  5  NHLe=0.189  95% CI: [0.123, 0.280]
19901991  n=  4  too few
19911992  n=  2  too few
19921993  n=  4  too few
19931994  n=  2  too few
19941995  n= 10  NHLe=0.214  95% CI: [0.157, 0.276]
19951996  n=  3  too few
19961997  n=  5  NHLe=0.119  95% CI: [0.073, 0.168]
19971998  n=  4  too few
19981999  n=  7  NHLe=0.240  95% CI: [0.186, 0.281]
19992000  n=  7  NHLe=0.189  95% CI: [0.117, 0.266]
20002001  n=  2  too few
20012002  n=  4  too few
20022003  n= 10  NHLe=0.230  95% CI: [0.173, 0.274]
20042005  n=  5  NHLe=0.260  95% CI: [0.189, 0.336]
20052006  n=  2  too few
20062007  n= 12  NHLe=0.259  95% CI: [0.186, 0.322]
20072008  n=  7  NHLe=0.299  95% CI: [0.247, 0.347]
20082009  n=  6  NHLe=0.386  95% CI: [0.299, 0.448]
20092010  n=  8  NHLe=0.338 

In [26]:
# --- OHL team goals per game, built from player totals ---
ohl = pd.read_sql("SELECT name, team_code, season_name, games_played, goals FROM ohl_skaters", conn)
ohl["season"] = ohl["season_name"].str[:4].astype(int) * 10000 + ohl["season_name"].str[:4].astype(int) + 1
ohl["name_key"] = ohl["name"].str.lower().str.strip()

teams = (ohl.groupby(["season", "team_code"])
            .agg(team_goals=("goals", "sum"), team_gp=("games_played", "max"), roster=("name", "count"))
            .reset_index())
teams = teams[teams["roster"] >= 15]            # drop odd combined/traded-player codes
teams["team_gpg"] = teams["team_goals"] / teams["team_gp"]
league_avg = teams.groupby("season")["team_gpg"].mean().rename("league_gpg")
teams = teams.merge(league_avg, on="season")

# --- Each player's team context (GP-weighted if he was traded) ---
ctx = ohl.merge(teams, on=["season", "team_code"])
ctx["w"] = ctx["games_played"]
ctx = (ctx.groupby(["name_key", "season"])
          .apply(lambda g: pd.Series({
              "team_gpg": np.average(g["team_gpg"], weights=g["w"]),
              "league_gpg": g["league_gpg"].iloc[0]}))
          .reset_index())

pairs["name_key"] = pairs["name"].str.lower().str.strip()
test = pairs.merge(ctx, left_on=["name_key", "ohl_season"], right_on=["name_key", "season"], how="inner")
print(f"Matched {len(test)} of {len(pairs)} pairs to OHL team data")

# --- Adjusted PPG: scale by league average / team scoring ---
test["adj_ohl_ppg"] = test["ohl_ppg"] * test["league_gpg"] / test["team_gpg"]

# --- Which predicts NHL PPG better? ---
def evaluate(col):
    factor = np.average(test["nhl_ppg"] / test[col], weights=test["nhl_gp"])   # not used for fit, just reference
    k = (test["nhl_pts"].sum() / test["nhl_gp"].sum()) / np.average(test[col], weights=test["ohl_gp"])
    pred = k * test[col]
    rmse = np.sqrt(np.average((test["nhl_ppg"] - pred) ** 2, weights=test["nhl_gp"]))
    corr = test[[col, "nhl_ppg"]].corr().iloc[0, 1]
    return k, rmse, corr

for col in ["ohl_ppg", "adj_ohl_ppg"]:
    k, rmse, corr = evaluate(col)
    print(f"{col:12}  factor={k:.3f}  RMSE={rmse:.4f}  correlation={corr:.3f}")


Matched 128 of 215 pairs to OHL team data
ohl_ppg       factor=0.291  RMSE=0.1485  correlation=0.622
adj_ohl_ppg   factor=0.334  RMSE=0.1602  correlation=0.547


In [27]:
unmatched = pairs[~pairs["name_key"].isin(test["name_key"]) & (pairs["ohl_season"] >= 20002001)]
print(len(unmatched), "unmatched since 2000")
unmatched[["name", "ohl_season"]].head(20)

pairs = pairs[pairs["ohl_season"] >= 20002001]

9 unmatched since 2000


In [ ]:
#testing if team gpg affects player performance and consequently NHLe
results = []
for alpha in np.arange(0, 1.01, 0.1):
    test["alpha_ppg"] = test["ohl_ppg"] * (test["league_gpg"] / test["team_gpg"]) ** alpha
    k, rmse, corr = evaluate("alpha_ppg")
    results.append({"alpha": round(alpha, 1), "factor": round(k, 3),
                    "rmse": round(rmse, 4), "corr": round(corr, 3)})

results = pd.DataFrame(results)
results

,alpha,factor,rmse,corr
0,0.0,0.291,0.1485,0.622
1,0.1,0.295,0.1487,0.620
2,0.2,0.299,0.1490,0.617
3,0.3,0.304,0.1496,0.613
4,0.4,0.308,0.1505,0.607
5,0.5,0.313,0.1515,0.600
6,0.6,0.317,0.1528,0.592
7,0.7,0.321,0.1543,0.583
8,0.8,0.326,0.1561,0.572
9,0.9,0.330,0.1580,0.560


In [38]:
pd.read_sql("""
SELECT league, COUNT(DISTINCT player_id) AS players
FROM career_seasons
WHERE league LIKE '%SHL%' OR league LIKE '%AHL%' OR league LIKE '%liiga%'
   OR league LIKE '%SM%' OR league LIKE '%NCAA%' OR league LIKE '%Swe%' OR league LIKE '%Fin%'
GROUP BY league ORDER BY players DESC
""", conn)

leagues = ["OHL", "WHL", "QMJHL", "AHL", "NCAA", "SHL", "Liiga"]   # edit to match Step 1

for league in leagues:
    p = build_pairs(league)
    if len(p) < 10:
        print(f"{league:8} n={len(p):4}  too few")
        continue
    low, high = np.percentile(bootstrap_nhle(p), [2.5, 97.5])
    print(f"{league:8} n={len(p):4}  NHLe={nhle(p):.3f}  95% CI: [{low:.3f}, {high:.3f}]")

OHL      n= 137  NHLe=0.290  95% CI: [0.271, 0.311]
WHL      n= 115  NHLe=0.296  95% CI: [0.266, 0.325]
QMJHL    n=  50  NHLe=0.289  95% CI: [0.255, 0.325]
AHL      n=2539  NHLe=0.486  95% CI: [0.476, 0.499]
NCAA     n= 109  NHLe=0.404  95% CI: [0.369, 0.439]
SHL      n=  48  NHLe=0.529  95% CI: [0.426, 0.632]
Liiga    n=  21  NHLe=0.512  95% CI: [0.376, 0.658]
